# Unit I: Data Preprocessing Pipeline

This notebook outlines the rigorous data cleaning workflow corresponding to Unit I of the project requirements.
Steps include:
1. Data Profiling & Collection
2. Missing Values Handling
3. Column Transformations & Renaming
4. Duplicate Handling
5. Encoding (Binary, Ordinal, and One-Hot)
6. Outlier Handling (IQR Method)
7. Feature Scaling (Standardization)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

### 1. Data Collection & Profiling
We begin by reading the raw CSV data and exploring its fundamental shape, schema, and basic statistics.

In [ ]:
df = pd.read_csv("../data/Customer_data.csv")
print("Initial Dataset Shape:", df.shape)
df.info()

### 2. Missing Values Handling
We identify missing values vertically across our features. According to best practices for this volume, sparse null records can be dropped.

In [ ]:
missing = df.isnull().sum()
print("Missing Values before cleaning:\n", missing[missing > 0])

# Dropping missing records
df = df.dropna()
print("\nShape after dropping NAs:", df.shape)

### 3. Column Transformations & Renaming
Column headers are standardized to lowercase snake_case to prevent coding errors. We also rename ambiguous columns for clarity.

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df = df.rename(columns={
    'no._of_orders_placed': 'demand',  # Handling the weird dot from raw data if present
    'no_of_orders_placed': 'demand',
    'restaurnat_rating': 'restaurant_rating'
})
print("Standardized Columns:", df.columns.tolist())

### 4. Duplicate Handling
We hunt for and eliminate exact duplicate rows to prevent bias.

In [ ]:
duplicates = df.duplicated().sum()
print(f"Found {duplicates} duplicate records.")

df = df.drop_duplicates()
print("Shape after dropping duplicates:", df.shape)

### 5. Fixing Data Types & Target Encoding (Ordinal / Binary)
We must clean erroneous string tokens out of numeric columns, convert binary categories, and ordinally map 1-5 scalar reviews.

In [ ]:
# Clean up 'delivery_time' logic error where the header text leaked into rows
df = df[df['delivery_time'] != 'Delivery Time']
df['delivery_time'] = df['delivery_time'].astype(int)

# Binary map gender
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

# Ordinal encodings mapping
mapping = {
    'Strongly disagree': 1,
    'Disagree': 2,
    'Neutral': 3,
    'Agree': 4,
    'Strongly agree': 5
}

ordinal_cols = [
    'ease_and_convenient',
    'self_cooking',
    'health_concern',
    'late_delivery',
    'poor_hygiene',
    'bad_past_experience',
    'more_offers_and_discount'
]

for col in ordinal_cols:
    df[col] = df[col].map(mapping)

### 6. Outlier Handling (IQR Method)
We will formally evaluate numerical dispersion using the Interquartile Range (IQR). Identifying statistical outliers allows us to explicitly decide to retain or cap them.

In [ ]:
numeric_cols = ['age', 'family_size', 'demand', 'delivery_time', 'order_value']

print("Outlier Detection Summary (IQR Method):")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col:15}: {len(outliers):3d} outliers detected.")

print("\nDecision: The identified outliers within continuous vectors represent naturally occurring extremes (like high order volumes or older ages) rather than collection errors. We will retain them to preserve data integrity.")

### 7. Categorical Encoding (One-Hot)
Converting remaining unstructured categoricals into dense binary vectors using `get_dummies` while dropping the first categorical to prevent severe multicollinearity.

In [ ]:
df = pd.get_dummies(df, drop_first=True)
print("Shape post One-Hot Encoding:", df.shape)

### 8. Feature Scaling 
Continuous variables operate on wildly different numerical scales (e.g., `demand` goes up to 250, while `order_value` stays at 1-3). We apply **StandardScaler** (Z-score normalization) to force these onto an equal playing field (Mean=0, Std=1).

In [ ]:
scaler = StandardScaler()
scale_cols = ['age', 'family_size', 'restaurant_rating', 'delivery_rating', 
              'demand', 'delivery_time', 'order_value']

# Scale the numeric columns
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("Scaling Verification (Mean ~ 0, Std ~ 1):")
display(df[scale_cols].describe().round(3).loc[['mean', 'std']])

### 9. Save Processed Object
We export the rigidly cleaned numerical structure for deployment into the secondary Descriptive Analytics notebook.

In [ ]:
df.to_csv("../data/cleaned_food_delivery_data.csv", index=False)
print("✓ Final pipeline execution complete. Cleaned and scaled data persisted successfully.")